In [ ]:

import cv2
import mediapipe as mp
import pandas as pd
import urllib.request
from pathlib import Path

In [ ]:
VIDEO_PATH        = "videoplayback.mp4"
SAIDA_CSV         = "dados_corpo_bola.csv"
SAIDA_VIDEO       = "video_anotado.mp4"

NUM_POSES         = 5
SCORE_BOLA        = 0.098
CONF_POSE         = 0.5

In [ ]:
PARTES = {
    "left_wrist":  "mao_esq",     "right_wrist":  "mao_dir",      # mão (pulso)
    "left_elbow":  "cotovelo_esq","right_elbow":  "cotovelo_dir", # cotovelo
    "left_hip":    "quadril_esq", "right_hip":    "quadril_dir",  # quadril
    "left_knee":   "joelho_esq",  "right_knee":   "joelho_dir",   # joelho
    "left_ankle":  "pe_esq",      "right_ankle":  "pe_dir",       # pé (tornozelo)
}

In [ ]:
# Nomes dos 33 landmarks do MediaPipe Pose (ordem fixa da API)
LANDMARK_NAMES = [
    "nose", "left_eye_inner", "left_eye", "left_eye_outer",
    "right_eye_inner", "right_eye", "right_eye_outer",
    "left_ear", "right_ear", "mouth_left", "mouth_right",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_pinky", "right_pinky",
    "left_index", "right_index", "left_thumb", "right_thumb",
    "left_hip", "right_hip", "left_knee", "right_knee",
    "left_ankle", "right_ankle", "left_heel", "right_heel",
    "left_foot_index", "right_foot_index",
]

# Conexões para desenhar o esqueleto no vídeo
CONEXOES = [
    ("left_shoulder", "right_shoulder"),
    ("left_shoulder", "left_elbow"),  ("left_elbow", "left_wrist"),
    ("right_shoulder", "right_elbow"),("right_elbow", "right_wrist"),
    ("left_shoulder", "left_hip"),    ("right_shoulder", "right_hip"),
    ("left_hip", "right_hip"),
    ("left_hip", "left_knee"),        ("left_knee", "left_ankle"),
    ("right_hip", "right_knee"),      ("right_knee", "right_ankle"),
]

In [ ]:
# ====================================================================
# 1) Baixa os modelos, se necessário
# ====================================================================
def baixar_modelo(caminho, url, nome):
    if not Path(caminho).exists():
        print(f"Baixando modelo de {nome}...")
        urllib.request.urlretrieve(url, caminho)
        print("  -> OK")
    else:
        print(f"Modelo de {nome} já existe.")


MODELO_BOLA = "efficientdet_lite0.tflite"
MODELO_POSE = "pose_landmarker_full.task"

baixar_modelo(
    MODELO_BOLA,
    "https://storage.googleapis.com/mediapipe-models/object_detector/"
    "efficientdet_lite0/int8/1/efficientdet_lite0.tflite",
    "bola",
)
baixar_modelo(
    MODELO_POSE,
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
    "pose_landmarker_full/float16/1/pose_landmarker_full.task",
    "pose",
)

In [ ]:

# ====================================================================
# 2) Abre o vídeo
# ====================================================================
if not Path(VIDEO_PATH).exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {VIDEO_PATH}")

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise ValueError("Erro ao abrir o vídeo.")

fps          = cap.get(cv2.CAP_PROP_FPS)
largura      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out = None
if SAIDA_VIDEO:
    out = cv2.VideoWriter(
        SAIDA_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps, (largura, altura)
    )

In [ ]:
# ====================================================================
# 3) Configura os dois detectores do MediaPipe
# ====================================================================
BaseOptions = mp.tasks.BaseOptions
VisionMode  = mp.tasks.vision.RunningMode

opcoes_bola = mp.tasks.vision.ObjectDetectorOptions(
    base_options=BaseOptions(model_asset_path=MODELO_BOLA),
    running_mode=VisionMode.VIDEO,
    max_results=10,
    score_threshold=SCORE_BOLA,
)

opcoes_pose = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODELO_POSE),
    running_mode=VisionMode.VIDEO,
    num_poses=NUM_POSES,
    min_pose_detection_confidence=CONF_POSE,
    min_pose_presence_confidence=CONF_POSE,
    min_tracking_confidence=CONF_POSE,
    output_segmentation_masks=False,
)

ObjectDetector  = mp.tasks.vision.ObjectDetector
PoseLandmarker  = mp.tasks.vision.PoseLandmarker

dados = []

In [ ]:

# ====================================================================
# 4) PASSAGEM ÚNICA pelo vídeo (bola + pose ao mesmo tempo)
# ====================================================================
with ObjectDetector.create_from_options(opcoes_bola) as det_bola, \
     PoseLandmarker.create_from_options(opcoes_pose) as det_pose:

    frame_num = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        tempo = frame_num / fps
        ts_ms = int(tempo * 1000)
        rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

        # ---------- DETECÇÃO DA BOLA ----------
        res_bola = det_bola.detect_for_video(mp_img, ts_ms)
        melhor_bola, melhor_score = None, -1
        for d in res_bola.detections:
            cat = d.categories[0]
            nome = (cat.category_name or "").lower()
            if "ball" in nome and cat.score > melhor_score:
                bb = d.bounding_box
                melhor_score = cat.score
                melhor_bola = {
                    "score": cat.score,
                    "x1": int(bb.origin_x), "y1": int(bb.origin_y),
                    "w": int(bb.width),     "h": int(bb.height),
                    "cx": int(bb.origin_x + bb.width / 2),
                    "cy": int(bb.origin_y + bb.height / 2),
                }

        bola_info = {
            "bola_detectada": melhor_bola is not None,
            "bola_x":         melhor_bola["cx"]    if melhor_bola else None,
            "bola_y":         melhor_bola["cy"]    if melhor_bola else None,
            "bola_confianca": round(melhor_bola["score"], 3) if melhor_bola else None,
        }

        # ---------- DETECÇÃO DA POSE ----------
        res_pose = det_pose.detect_for_video(mp_img, ts_ms)

        if res_pose.pose_landmarks:
            for pessoa_id, landmarks in enumerate(res_pose.pose_landmarks):
                reg = {"frame": frame_num, "tempo_segundos": round(tempo, 3)}
                reg.update(bola_info)
                reg["pessoa_id"]      = pessoa_id
                reg["pose_detectada"] = True

                # Mapa nome -> landmark p/ acesso rápido
                lm_por_nome = {LANDMARK_NAMES[i]: lm for i, lm in enumerate(landmarks)}

                # Só as partes pedidas
                for nome_lm, rotulo in PARTES.items():
                    lm = lm_por_nome[nome_lm]
                    reg[f"{rotulo}_x"]   = round(lm.x * largura, 2)
                    reg[f"{rotulo}_y"]   = round(lm.y * altura, 2)
                    reg[f"{rotulo}_vis"] = round(lm.visibility, 3)

                dados.append(reg)

                # ---------- desenha no vídeo ----------
                if out is not None:
                    pts = {
                        LANDMARK_NAMES[i]: (int(lm.x * largura), int(lm.y * altura))
                        for i, lm in enumerate(landmarks)
                    }
                    for a, b in CONEXOES:
                        cv2.line(frame, pts[a], pts[b], (0, 255, 0), 2)
                    for rotulo_lm, coord in pts.items():
                        cv2.circle(frame, coord, 4, (255, 255, 0), -1)
        else:
            # frame sem pose
            reg = {"frame": frame_num, "tempo_segundos": round(tempo, 3)}
            reg.update(bola_info)
            reg["pessoa_id"]      = None
            reg["pose_detectada"] = False
            dados.append(reg)

        # ---------- desenha a bola no vídeo ----------
        if out is not None and melhor_bola:
            x1, y1 = melhor_bola["x1"], melhor_bola["y1"]
            w, h   = melhor_bola["w"], melhor_bola["h"]
            cv2.rectangle(frame, (x1, y1), (x1 + w, y1 + h), (0, 255, 255), 3)
            cv2.circle(frame, (melhor_bola["cx"], melhor_bola["cy"]), 5, (0, 0, 255), -1)
            cv2.putText(frame, f"Bola {melhor_bola['score']:.2f}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        if out is not None:
            out.write(frame)

        frame_num += 1
        if frame_num % 50 == 0:
            print(f"Processando frame {frame_num}/{total_frames}")

cap.release()
if out is not None:
    out.release()

In [ ]:
# ====================================================================
# 5) Salva o CSV
# ====================================================================
# Ordem amigável das colunas
colunas = ["frame", "tempo_segundos", "pessoa_id", "pose_detectada",
           "bola_detectada", "bola_x", "bola_y", "bola_confianca"]
for rotulo in PARTES.values():
    colunas += [f"{rotulo}_x", f"{rotulo}_y", f"{rotulo}_vis"]

df = pd.DataFrame(dados)
df = df.reindex(columns=colunas)   # garante a ordem mesmo em frames sem pose
df.to_csv(SAIDA_CSV, index=False)

print("\nConcluído!")
print(f"CSV salvo em:   {SAIDA_CSV}  ({len(df)} linhas, {len(df.columns)} colunas)")
if SAIDA_VIDEO:
    print(f"Vídeo salvo em: {SAIDA_VIDEO}")
print(df.head())